In [ ]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'

# embryo_datasets = {
#     # 25C embryos Var2, 0.173 um/pixel
#     '001': [
#         'test_data/NSPARC/2025-03-31/MCP-mSG_His-RFP_Var2(001)_embryo01',
#         'test_data/NSPARC/2025-03-31/MCP-mSG_His-RFP_Var2(001)_embryo02',
#         'test_data/NSPARC/2025-04-01/MCP-mSG_His-RFP_Var2(001)_embryo20',
#         'test_data/NSPARC/2025-04-01/MCP-mSG_His-RFP_Var2(001)_embryo38',
#         'test_data/NSPARC/2025-04-14/MCP-mSG_His-RFP_Var2(001)_embryo28',
#         'test_data/NSPARC/2025-04-15/MCP-mSG_His-RFP_Var2(001)_embryo01',
#     ],
#
#     # 22C embryos PWM, 0.173 um/pixel
#     '003': [
#         'test_data/NSPARC/2025-04-29/MCP-mSG_His-RFP_RBSPWM(003)_embryo01',
#         'test_data/NSPARC/2025-04-29/MCP-mSG_His-RFP_RBSPWM(003)_embryo02',
#         'test_data/NSPARC/2025-04-29/MCP-mSG_His-RFP_RBSPWM(003)_embryo03',
#         'test_data/NSPARC/2025-05-06/MCP-mSG_His-RFP_RBSPWM(003)_embryo01_22C',
#         'test_data/NSPARC/2025-05-09/MCP-mSG_His-RFP_PWM(003)_embryo01',
#     ],
# }

embryo_datasets = {
    # 22C embryos Var2, 0.230 um/pixel
    '001': [
        'test_data/NSPARC/2025-05-28/MCP-mSG_His-RFP_RBS(001)_embryo01',
        'test_data/NSPARC/2025-05-28/MCP-mSG_His-RFP_RBS(001)_embryo02',
        'test_data/NSPARC/2025-05-28/MCP-mSG_His-RFP_RBS(001)_embryo03',
        'test_data/NSPARC/2025-05-28/MCP-mSG_His-RFP_RBS(001)_embryo04',
    ],

    # 22C embryos PWM, 0.230 um/pixel
    '003': [
        'test_data/NSPARC/2025-05-27/MCP-mSG_His-RFP_RBS(003)_embryo02',
        'test_data/NSPARC/2025-05-29/MCP-mSG_His-RFP_RBS(003)_embryo01',
        'test_data/NSPARC/2025-05-29/MCP-mSG_His-RFP_RBS(003)_embryo02',
    ],
}

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# Define the AP range of interest
ap_range = (0.15, 0.4)

# Create a dictionary to group embryos by group
grouped = average_fit_df.groupby('group')

group_averages = {}

for group_name, group_df in grouped:
    plt.figure(figsize=(10, 6))
    all_x, all_y, all_yerr = [], [], []

    for _, row in group_df.iterrows():
        embryo_id = row['embryo_id']
        embryo_path = os.path.join(embryo_id, 'bin_fits_checked.pkl')

        if not os.path.exists(embryo_path):
            print(f"Missing file: {embryo_path}")
            continue

        df = pd.read_pickle(embryo_path)

        x = np.array(df['ap_bin'])
        y = np.array(df['bin_fit_slope'])
        y_err = np.array(df.get('bin_fit_slope_err', np.full_like(y, np.nan)))  # Use if available

        plt.errorbar(x, y, yerr=y_err, fmt='o', alpha=0.6, label=os.path.basename(embryo_id))
        all_x.append(x)
        all_y.append(y)
        all_yerr.append(y_err)

    # Combine data by AP bin
    bin_dict = {}
    err_dict = {}

    for x_vals, y_vals, yerr_vals in zip(all_x, all_y, all_yerr):
        for xi, yi, yerri in zip(x_vals, y_vals, yerr_vals):
            xi_rounded = np.round(xi, 3)
            bin_dict.setdefault(xi_rounded, []).append(yi)
            err_dict.setdefault(xi_rounded, []).append(yerri)

    avg_bins = sorted(bin_dict.keys())

    # Mean and standard error propagation
    avg_rates = []
    avg_errors = []
    for b in avg_bins:
        vals = np.array(bin_dict[b])
        errs = np.array(err_dict[b])
        n = len(vals)
        mean = np.nanmean(vals)
        se = np.sqrt(np.nansum(errs ** 2)) / n  # Error propagation
        avg_rates.append(mean)
        avg_errors.append(se)

    group_averages[group_name] = (avg_bins, avg_rates, avg_errors)

    # Plot group average
    plt.plot(avg_bins, avg_rates, '-o', color='black', label=f'{group_name} Average')
    plt.fill_between(avg_bins,
                     np.array(avg_rates) - np.array(avg_errors),
                     np.array(avg_rates) + np.array(avg_errors),
                     color='black', alpha=0.2)

    # Highlight AP range of interest
    plt.axvspan(ap_range[0], ap_range[1], color='gray', alpha=0.15, label='AP range 0.15–0.4')

    # Compute and display mean over AP range
    ap_vals = np.array(avg_bins)
    rate_vals = np.array(avg_rates)
    range_mask = (ap_vals >= ap_range[0]) & (ap_vals <= ap_range[1])
    mean_in_range = np.nanmean(rate_vals[range_mask])
    print(f'{group_name}: Mean rate in AP range {ap_range} = {mean_in_range:.3f} AU')

    # Plot formatting
    plt.title(f'Slope Data: {group_name}')
    plt.xlabel('AP Position')
    plt.ylabel('bin_fit_slope (AU/min)')
    plt.ylim(bottom=0)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Combined group averages ---
plt.figure(figsize=(10, 6))

for group_name, (avg_bins, avg_rates, avg_errors) in group_averages.items():
    plt.plot(avg_bins, avg_rates, '-o', label=f'{group_name} Avg')
    plt.fill_between(avg_bins,
                     np.array(avg_rates) - np.array(avg_errors),
                     np.array(avg_rates) + np.array(avg_errors),
                     alpha=0.2)

# Highlight AP range
plt.axvspan(ap_range[0], ap_range[1], color='gray', alpha=0.15, label='AP range 0.15–0.4')

plt.title('Group Averages Overlay')
plt.xlabel('AP Position')
plt.ylabel('bin_fit_slope (AU/min)')
plt.legend()
plt.ylim(bottom=0)
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'grouped' is not defined